# Project 2 — CORES for Monocular Depth Estimation

Kaggle-first implementation for studying convolutional-response OOD detection in a monocular depth estimation network.

**Planned domains:** NYU Depth v2 (ID) and KITTI (OOD).  
**Planned model:** FastDepth.  
**Main metrics:** AUROC, FPR95, RMSE, AbsRel, δ1, δ2, δ3.

> Start with `QUICK_MODE = True`. Dataset-specific code will be enabled after the exact Kaggle dataset sources and layouts have been selected.

## 1. Imports

In [ ]:
from __future__ import annotations

import json
import itertools
import os
import platform
import random
import sys
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')

## 2. Environment and configuration

In [ ]:
def detect_environment() -> str:
    if Path('/kaggle').exists():
        return 'kaggle'
    if 'google.colab' in sys.modules:
        return 'colab'
    return 'local'


ENVIRONMENT = detect_environment()

if ENVIRONMENT == 'kaggle':
    INPUT_ROOT = Path('/kaggle/input')
    WORK_ROOT = Path('/kaggle/working/cores-mde')
elif ENVIRONMENT == 'colab':
    INPUT_ROOT = Path('/content/data')
    WORK_ROOT = Path('/content/cores-mde')
else:
    INPUT_ROOT = Path.cwd().parent / 'data'
    WORK_ROOT = Path.cwd().parent / 'outputs'

WORK_ROOT.mkdir(parents=True, exist_ok=True)
(WORK_ROOT / 'checkpoints').mkdir(exist_ok=True)
(WORK_ROOT / 'figures').mkdir(exist_ok=True)
(WORK_ROOT / 'results').mkdir(exist_ok=True)

print(f'Environment: {ENVIRONMENT}')
print(f'Input root: {INPUT_ROOT}')
print(f'Work root: {WORK_ROOT}')

In [ ]:
@dataclass(frozen=True)
class Config:
    seed: int = 42
    quick_mode: bool = True
    train_model: bool = False
    load_checkpoint: bool = True
    image_height: int = 224
    image_width: int = 304
    batch_size: int = 8
    num_workers: int = 2
    epochs: int = 2
    learning_rate: float = 1e-4
    nyu_dataset_dir: str = 'datasets/awsaf49/nyuv2-official-split-dataset'
    kitti_dataset_dir: str = 'datasets/artemmmtry/kitti-depth-prediction-evaluation'


CFG = Config()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def validate_cuda_compatibility() -> None:
    if not torch.cuda.is_available():
        return
    major, minor = torch.cuda.get_device_capability(0)
    device_arch = f'sm_{major}{minor}'
    supported_arches = set(torch.cuda.get_arch_list())
    if supported_arches and device_arch not in supported_arches:
        raise RuntimeError(
            f'GPU architecture {device_arch} is not supported by this PyTorch build. ' +
            'On Kaggle, switch from P100 to GPU T4 x2 and restart the session.'
        )


print(json.dumps(asdict(CFG), indent=2))
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'CUDA: {torch.version.cuda}')
    print(f'GPU count: {torch.cuda.device_count()}')
    validate_cuda_compatibility()
else:
    print('WARNING: GPU not detected. Enable a GPU accelerator for training.')

## 3. Reproducibility

In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

    # Determinism improves reproducibility but may reduce performance.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(CFG.seed)
print(f'Global seed set to {CFG.seed}.')

## 4. Runtime diagnostics

In [ ]:
def runtime_report() -> dict[str, Any]:
    report: dict[str, Any] = {
        'environment': ENVIRONMENT,
        'platform': platform.platform(),
        'python': sys.version.split()[0],
        'torch': torch.__version__,
        'numpy': np.__version__,
        'sklearn': sklearn.__version__,
        'device': str(DEVICE),
        'cuda_available': torch.cuda.is_available(),
    }
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        report.update({
            'gpu': props.name,
            'gpu_memory_gib': round(props.total_memory / 1024**3, 2),
            'cuda': torch.version.cuda,
            'gpu_compute_capability': '.'.join(map(str, torch.cuda.get_device_capability(0))),
            'pytorch_cuda_arch_list': torch.cuda.get_arch_list(),
        })
    return report


RUNTIME = runtime_report()
print(json.dumps(RUNTIME, indent=2))
with (WORK_ROOT / 'runtime.json').open('w', encoding='utf-8') as file:
    json.dump(RUNTIME, file, indent=2)

## 5. Dataset discovery

This section deliberately fails early when dataset names have not been configured. It prevents a long Kaggle run from silently using the wrong folders.

In [ ]:
def list_input_datasets(root: Path) -> list[Path]:
    if not root.exists():
        return []
    return sorted(path for path in root.iterdir() if path.is_dir())


available_datasets = list_input_datasets(INPUT_ROOT)
print('Attached input datasets:')
for path in available_datasets:
    print(f'  - {path.name}')
if not available_datasets:
    print('  (none found — expected before Kaggle datasets are attached)')

In [ ]:
def resolve_dataset_root(input_root: Path, dataset_dir: str) -> Path:
    direct_path = input_root / dataset_dir
    if direct_path.is_dir():
        return direct_path

    # Kaggle may mount inputs as /kaggle/input/datasets/<owner>/<slug>.
    matches = sorted(
        path for path in input_root.rglob(dataset_dir) if path.is_dir()
    ) if input_root.exists() else []
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise RuntimeError(
            f'Multiple folders match {dataset_dir!r}: {[str(path) for path in matches]}'
        )
    return direct_path


NYU_ROOT = resolve_dataset_root(INPUT_ROOT, CFG.nyu_dataset_dir)
KITTI_ROOT = resolve_dataset_root(INPUT_ROOT, CFG.kitti_dataset_dir)

dataset_status = pd.DataFrame([
    {'dataset': 'NYU Depth v2', 'path': str(NYU_ROOT), 'found': NYU_ROOT.exists()},
    {'dataset': 'KITTI', 'path': str(KITTI_ROOT), 'found': KITTI_ROOT.exists()},
])
display(dataset_status)

DATASETS_READY = bool(dataset_status['found'].all())
if not DATASETS_READY:
    print('Dataset loaders remain disabled until the Config folder names are updated.')

In [ ]:
def compact_dataset_manifest(root: Path, max_examples: int = 30) -> pd.DataFrame:
    if not root.exists():
        return pd.DataFrame(columns=['relative_path', 'suffix', 'size_mib'])
    files = list(itertools.islice(
        (path for path in root.rglob('*') if path.is_file()), max_examples
    ))
    rows = [
        {
            'relative_path': str(path.relative_to(root)),
            'suffix': path.suffix.lower(),
            'size_mib': round(path.stat().st_size / 1024**2, 3),
        }
        for path in files
    ]
    print(f'{root.name}: showing up to {max_examples} discovered files')
    return pd.DataFrame(rows)


if NYU_ROOT.exists():
    display(compact_dataset_manifest(NYU_ROOT))
if KITTI_ROOT.exists():
    display(compact_dataset_manifest(KITTI_ROOT))

## 6. Core validation utilities

### NYU Depth v2 loader

RGB and depth files are paired by their shared identifier. Encoded uint16 depth values are mapped to the documented 0–10 metre interval. Depth remains at native resolution for evaluation.

In [ ]:
NYU_UINT16_MAX = float(2**16 - 1)
NYU_MAX_DEPTH_METERS = 10.0


def find_nyu_split_directory(root: Path, split: str) -> Path:
    if split == 'train':
        split_dir = root / 'train'
    elif split in {'test', 'val', 'validation'}:
        split_dir = root / 'test' / 'official'
    else:
        raise ValueError("split must be 'train' or 'test'.")
    if not split_dir.is_dir():
        raise FileNotFoundError(f'NYU split directory not found: {split_dir}')
    return split_dir


def discover_nyu_pairs(root: Path, split: str):
    split_dir = find_nyu_split_directory(root, split)
    rgb_paths = sorted(split_dir.rglob('rgb_*.png'))
    pairs = [
        (rgb, rgb.with_name(rgb.name.replace('rgb_', 'depth_', 1)))
        for rgb in rgb_paths
    ]
    missing = [depth for _, depth in pairs if not depth.is_file()]
    if missing or not pairs:
        raise FileNotFoundError(
            f'Invalid NYU pairing: {len(pairs)} pairs, {len(missing)} missing depths.'
        )
    return pairs


class NYUDepthDataset(Dataset):
    def __init__(self, root: Path, split: str, output_size=(224, 304), limit=None):
        self.output_size = output_size
        pairs = discover_nyu_pairs(root, split)
        self.pairs = pairs[:limit] if limit is not None else pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        image_path, depth_path = self.pairs[index]
        with Image.open(image_path) as image_file:
            original_size = (image_file.height, image_file.width)
            image = rgb_to_tensor(image_file, self.output_size)
        with Image.open(depth_path) as depth_file:
            encoded_depth = np.asarray(depth_file, dtype=np.float32)
        depth = torch.from_numpy(
            encoded_depth / NYU_UINT16_MAX * NYU_MAX_DEPTH_METERS
        ).unsqueeze(0)
        return {
            'image': image,
            'depth': depth,
            'valid_mask': torch.isfinite(depth) & (depth > 0) & (depth <= 10.0),
            'original_size': original_size,
            'sample_id': image_path.stem.removeprefix('rgb_'),
        }

### KITTI validation loader

KITTI depth PNG values are converted to metres by dividing by 256. Zero-valued pixels remain invalid. RGB is resized for the network, while depth stays at native resolution so predictions can later be resized back for evaluation.

In [ ]:
KITTI_DEPTH_SCALE = 256.0


def find_kitti_validation_directories(root: Path) -> tuple[Path, Path]:
    candidates = sorted(
        path for path in root.rglob('val_selection_cropped') if path.is_dir()
    )
    if len(candidates) != 1:
        raise FileNotFoundError(
            f'Expected one val_selection_cropped below {root}, found {len(candidates)}.'
        )
    image_dir = candidates[0] / 'image'
    depth_dir = candidates[0] / 'groundtruth_depth'
    if not image_dir.is_dir() or not depth_dir.is_dir():
        raise FileNotFoundError(f'Incomplete KITTI selection below {candidates[0]}.')
    return image_dir, depth_dir


def rgb_to_tensor(image: Image.Image, output_size: tuple[int, int]) -> torch.Tensor:
    array = np.asarray(image.convert('RGB'), dtype=np.float32) / 255.0
    tensor = torch.from_numpy(array).permute(2, 0, 1).unsqueeze(0)
    return F.interpolate(
        tensor, size=output_size, mode='bilinear', align_corners=False
    ).squeeze(0)


class KITTIDepthValidationDataset(Dataset):
    def __init__(self, root: Path, output_size=(224, 304), limit=None):
        self.output_size = output_size
        image_dir, depth_dir = find_kitti_validation_directories(root)
        images = sorted(image_dir.glob('*.png'))
        pairs = [(path, depth_dir / path.name) for path in images]
        missing = [depth for _, depth in pairs if not depth.is_file()]
        if missing or not pairs:
            raise FileNotFoundError(
                f'Invalid KITTI pairing: {len(pairs)} pairs, {len(missing)} missing depths.'
            )
        self.pairs = pairs[:limit] if limit is not None else pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        image_path, depth_path = self.pairs[index]
        with Image.open(image_path) as image_file:
            original_size = (image_file.height, image_file.width)
            image = rgb_to_tensor(image_file, self.output_size)
        with Image.open(depth_path) as depth_file:
            depth = torch.from_numpy(
                np.asarray(depth_file, dtype=np.float32) / KITTI_DEPTH_SCALE
            ).unsqueeze(0)
        return {
            'image': image,
            'depth': depth,
            'valid_mask': depth > 0,
            'original_size': original_size,
            'sample_id': image_path.stem,
        }

In [ ]:
if KITTI_ROOT.exists():
    kitti_limit = 16 if CFG.quick_mode else None
    kitti_dataset = KITTIDepthValidationDataset(
        KITTI_ROOT,
        output_size=(CFG.image_height, CFG.image_width),
        limit=kitti_limit,
    )
    kitti_sample = kitti_dataset[0]
    validate_later = (kitti_sample['image'], kitti_sample['depth'])
    print(f'KITTI samples ready: {len(kitti_dataset)}')
    print(f"RGB input: {tuple(kitti_sample['image'].shape)}")
    print(f"Native depth: {tuple(kitti_sample['depth'].shape)}")
else:
    kitti_dataset = None
    print('Attach the KITTI Kaggle input to enable this loader.')

In [ ]:
if NYU_ROOT.exists():
    train_limit = 32 if CFG.quick_mode else None
    test_limit = 16 if CFG.quick_mode else None
    nyu_train_dataset = NYUDepthDataset(
        NYU_ROOT, 'train', (CFG.image_height, CFG.image_width), train_limit
    )
    nyu_test_dataset = NYUDepthDataset(
        NYU_ROOT, 'test', (CFG.image_height, CFG.image_width), test_limit
    )
    nyu_sample = nyu_test_dataset[0]
    print(f'NYU train samples ready: {len(nyu_train_dataset)}')
    print(f'NYU test samples ready: {len(nyu_test_dataset)}')
    print(f"RGB input: {tuple(nyu_sample['image'].shape)}")
    print(f"Native depth: {tuple(nyu_sample['depth'].shape)}")
    print(
        f"Valid depth range: {nyu_sample['depth'][nyu_sample['valid_mask']].min():.3f}–"
        f"{nyu_sample['depth'][nyu_sample['valid_mask']].max():.3f} m"
    )
else:
    nyu_train_dataset = nyu_test_dataset = None
    print('Attach the NYU Kaggle input to enable this loader.')

In [ ]:
if nyu_test_dataset is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].imshow(nyu_sample['image'].permute(1, 2, 0).numpy())
    axes[0].set_title('NYU RGB (network input)')
    depth_view = axes[1].imshow(nyu_sample['depth'].squeeze(0).numpy(), cmap='magma')
    axes[1].set_title('NYU depth (metres, native size)')
    fig.colorbar(depth_view, ax=axes[1], label='metres')
    for axis in axes:
        axis.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
def validate_rgb_depth_pair(image: torch.Tensor, depth: torch.Tensor) -> None:
    if image.ndim != 3 or image.shape[0] != 3:
        raise ValueError(f'Expected RGB tensor [3, H, W], got {tuple(image.shape)}')
    if depth.ndim not in (2, 3):
        raise ValueError(f'Expected depth tensor [H, W] or [1, H, W], got {tuple(depth.shape)}')
    depth_hw = depth.shape[-2:]
    if image.shape[-2:] != depth_hw:
        raise ValueError(f'RGB/depth spatial mismatch: {image.shape[-2:]} vs {depth_hw}')
    if not torch.isfinite(image).all():
        raise ValueError('RGB tensor contains NaN or infinity.')
    if not torch.isfinite(depth).all():
        raise ValueError('Depth tensor contains NaN or infinity.')


dummy_image = torch.rand(3, CFG.image_height, CFG.image_width)
dummy_depth = torch.rand(1, CFG.image_height, CFG.image_width)
validate_rgb_depth_pair(dummy_image, dummy_depth)
print('RGB/depth validation smoke test passed.')

## 7. Depth-estimation metrics

In [ ]:
DEPTH_METRIC_NAMES = ('rmse', 'abs_rel', 'delta1', 'delta2', 'delta3')


def _as_bhw(tensor: torch.Tensor, name: str) -> torch.Tensor:
    if tensor.ndim == 2:
        return tensor.unsqueeze(0)
    if tensor.ndim == 3:
        return tensor
    if tensor.ndim == 4 and tensor.shape[1] == 1:
        return tensor[:, 0]
    raise ValueError(
        f'{name} must have shape [H, W], [B, H, W], or [B, 1, H, W]; ' +
        f'got {tuple(tensor.shape)}'
    )


@torch.no_grad()
def compute_depth_metrics(
    prediction: torch.Tensor,
    target: torch.Tensor,
    *,
    min_depth: float = 1e-3,
    max_depth: float = 10.0,
) -> dict[str, float]:
    if min_depth <= 0 or max_depth <= min_depth:
        raise ValueError('Expected 0 < min_depth < max_depth.')

    prediction = _as_bhw(prediction.detach(), 'prediction').float()
    target = _as_bhw(target.detach(), 'target').float()
    if prediction.shape != target.shape:
        raise ValueError(
            f'Prediction/target shape mismatch: {tuple(prediction.shape)} vs ' +
            f'{tuple(target.shape)}'
        )

    per_image = {name: [] for name in DEPTH_METRIC_NAMES}
    for predicted_depth, target_depth in zip(prediction, target):
        valid = (
            torch.isfinite(predicted_depth)
            & torch.isfinite(target_depth)
            & (target_depth >= min_depth)
            & (target_depth <= max_depth)
        )
        if not torch.any(valid):
            continue

        pred = predicted_depth[valid].clamp(min=min_depth, max=max_depth)
        true = target_depth[valid]
        ratio = torch.maximum(true / pred, pred / true)
        per_image['rmse'].append(torch.sqrt(torch.mean((pred - true) ** 2)))
        per_image['abs_rel'].append(torch.mean(torch.abs(pred - true) / true))
        per_image['delta1'].append(torch.mean((ratio < 1.25).float()))
        per_image['delta2'].append(torch.mean((ratio < 1.25**2).float()))
        per_image['delta3'].append(torch.mean((ratio < 1.25**3).float()))

    if not per_image['rmse']:
        raise ValueError('The batch does not contain any valid depth pixels.')
    return {
        name: torch.stack(values).mean().item()
        for name, values in per_image.items()
    }

In [ ]:
# Deterministic smoke tests: these must pass before dataset evaluation.
perfect_target = torch.tensor([[[[1.0, 2.0], [4.0, 8.0]]]])
perfect_metrics = compute_depth_metrics(perfect_target.clone(), perfect_target)
assert perfect_metrics['rmse'] == 0.0
assert perfect_metrics['abs_rel'] == 0.0
assert perfect_metrics['delta1'] == 1.0
assert perfect_metrics['delta2'] == 1.0
assert perfect_metrics['delta3'] == 1.0

scaled_target = torch.ones(1, 1, 2, 2)
scaled_metrics = compute_depth_metrics(scaled_target * 2.0, scaled_target)
assert np.isclose(scaled_metrics['rmse'], 1.0)
assert np.isclose(scaled_metrics['abs_rel'], 1.0)
assert scaled_metrics['delta1'] == 0.0
assert scaled_metrics['delta2'] == 0.0
assert scaled_metrics['delta3'] == 0.0

display(pd.DataFrame([perfect_metrics, scaled_metrics], index=['perfect', '2x scale']))
print('Depth metric smoke tests passed.')

## 8. Next implementation milestone

After selecting the exact Kaggle dataset sources:

1. inspect their real directory layouts and metadata;
2. implement NYU and KITTI `Dataset` classes;
3. visualize RGB/depth pairs and valid-depth masks;
4. implement and test the depth metrics;
5. integrate FastDepth and overfit a single batch;
6. add CORES only after the depth baseline is verified.